# Pipeline from a 1D spectrum

Simulate a BSPEC rate map from a BT-Settl template, build a `RampModel`, sync BSPEC calibrations from KOA if needed, then run `Stage0Pipeline` → `Stage1Pipeline` → `Stage2Pipeline` with defaults and show the stellar-RV preview.

**Environment variables** (set these in your shell / notebook kernel before running):

| Variable | Required when | Purpose |
|---|---|---|
| `KOA_CALIBRATION_CACHE` | Always | Local KOA calibration cache directory (and SQLite DB) |
| `KECK_OBSERVER_EMAIL` | Syncing / downloading from KOA | Keck Observer account email |
| `KECK_OBSERVER_PASSWORD` | Syncing / downloading from KOA | Keck Observer account password |

If `KOA_CALIBRATION_CACHE` is unset, the setup cell sets it under `OUT`.  
The KOA cell below syncs metadata from remote KOA and downloads any **BSPEC** calibration files that are not already in the cache.


In [1]:
import os

# OUTPUT DIRECTORY — update this to your desired output directory
OUT = "/Users/ckilpatrick/Downloads/hispec_pipeline_example"
os.makedirs(OUT, exist_ok=True)

os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    os.path.join(OUT, "cals", "KOA_CALIBRATION_CACHE"),
)
print("output:", OUT)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])


output: /Users/ckilpatrick/Downloads/hispec_pipeline_example
KOA_CALIBRATION_CACHE: /Users/ckilpatrick/Downloads/hispec_pipeline_example/cals/KOA_CALIBRATION_CACHE


## 1. Sync / verify KOA caldb (BSPEC)


In [2]:
from hispecdrp.calibrations import verify_koa_calibrations

# Requires env vars: KOA_CALIBRATION_CACHE, KECK_OBSERVER_EMAIL, KECK_OBSERVER_PASSWORD
SPECTROGRAPH = "BSPEC"
summary = verify_koa_calibrations(SPECTROGRAPH)

print(
    f"{summary['spectrograph']} caldb: {summary['n_records']} record(s); "
    f"downloaded {summary['n_downloaded']}/{summary['n_missing']} missing file(s)"
)
print("types:", summary["cal_types"])
if summary["downloaded"]:
    print("downloaded:", summary["downloaded"])
if summary["failed"]:
    print("failed:", summary["failed"])


[2026-07-25T15:54:27][koa_middleware.store:186][INFO]: Cache initialized: cache_dir='/Users/ckilpatrick/Downloads/hispec_pipeline_example/cals/KOA_CALIBRATION_CACHE', data_dir='/Users/ckilpatrick/Downloads/hispec_pipeline_example/cals/KOA_CALIBRATION_CACHE/calibrations/hispec/', db='/Users/ckilpatrick/Downloads/hispec_pipeline_example/cals/KOA_CALIBRATION_CACHE/database/hispec_calibrations.db', table='hispec'


[2026-07-25T15:54:27][koa_middleware.keck_client:81][INFO]: Session validation: no observer ID found; re-login required.


[2026-07-25T15:54:27][koa_middleware.keck_client:51][INFO]: No valid login detected, logging in...


[2026-07-25T15:54:30][koa_middleware.keck_client:136][INFO]: User info request successful: observer ID=4187


[2026-07-25T15:54:30][koa_middleware.keck_client:56][INFO]: Keck Observer login successful! Observer ID = 4187


[2026-07-25T15:54:30][koa_middleware.database.remote_database:35][INFO]: RemoteCalibrationDB initialized: instrument='hispec', url='https://www3.keck.hawaii.edu/api/calibrations'


... (4 outputs left out)

[2026-07-25T15:54:30][hispecdrp.calibrations.koa_sync:64][INFO]: Synced 0 KOA calibration record(s) into the local caldb.


[2026-07-25T15:54:30][koa_middleware.store:468][WARNING]: Found 1 calibration file(s) recorded in the DB but missing from the local cache.


[2026-07-25T15:54:30][hispecdrp.calibrations.koa_sync:114][INFO]: BSPEC caldb: 13 record(s); downloaded 0/0 missing file(s) (0 failed); types=['bias', 'dark', 'detflat', 'dq', 'fiberflat', 'gain', 'lfc_wavesol', 'nonlin', 'rn', 'saturation', 'sim_wavesol', 'tracepos', 'traceprof']


BSPEC caldb: 13 record(s); downloaded 0/0 missing file(s)
types: ['bias', 'dark', 'detflat', 'dq', 'fiberflat', 'gain', 'lfc_wavesol', 'nonlin', 'rn', 'saturation', 'sim_wavesol', 'tracepos', 'traceprof']


## 2. Simulated rate map from a 1D spectrum → `RampModel`

Load a BT-Settl template, create a rate map with `SimulateRateMap`, then build a raw UTR `RampModel` and provide the caldb files that were used to create the physical ramp.


In [3]:
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum

FIBERS = [1, 2, 3]  # SPECTROGRAPH is set in the KOA cell above

# Build a rate map from a 1D BT-Settl template and a raw UTR RampModel (KOA masters from cache).
ramp, cals, rate_map = create_ramp_from_spectrum(
    OUT,
    spectrograph=SPECTROGRAPH,
    fibers=FIBERS,
    template_filename="lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.h5",
    mag_ab=3.0,  # brightest non-saturating for these params (~78% full well; J=2.5 saturates)
    filter_band="J",  # BSPEC (yJ)
)

# Data models expose class name, filename, and arrays as attributes:
print(f"ramp type: {ramp.meta.model_type}")
print(f"ramp filename: {ramp.meta.filename}")
print(f"ramp data shape: {ramp.data.shape}")
print(f"Stage-0 cals: {sorted(cals)}")
print(f"rate_map shape: {rate_map.shape}")


[2026-07-25T15:54:32][hispecdrp.simulations.simulate_up_the_ramp:966][INFO]: Building default rate map (fibers=[1, 2, 3], spectrograph=BSPEC, output_dir=/Users/ckilpatrick/Downloads/hispec_pipeline_example).


[2026-07-25T15:54:32][hispecdrp.utils.spectral_templates_utils:486][INFO]: Normalized spectrum to J=3.00 (AB mag); scale=6.0900e+04 (was 14.96 mag)


[2026-07-25T15:54:32][hispecdrp.simulations.simulate_rate_map:146][WARNING]: Input spectrum does not have sufficient sampling for HISPEC simulation. The input spectrum will be interpolated to a higher sampling. If the input is intended to be a list of unresolved lines, or you do not want to interpolate to higher sampling, set continuous=False.


[2026-07-25T15:54:32][hispecdrp.simulations.simulate_rate_map:146][WARNING]: Input spectrum does not have sufficient sampling for HISPEC simulation. The input spectrum will be interpolated to a higher sampling. If the input is intended to be a list of unresolved lines, or you do not want to interpolate to higher sampling, set continuous=False.


[2026-07-25T15:54:32][hispecdrp.simulations.simulate_rate_map:146][WARNING]: Input spectrum does not have sufficient sampling for HISPEC simulation. The input spectrum will be interpolated to a higher sampling. If the input is intended to be a list of unresolved lines, or you do not want to interpolate to higher sampling, set continuous=False.


[2026-07-25T15:54:33][hispecdrp.simulations.simulate_rate_map:482][INFO]: Creating native rate map (spectrograph=BSPEC, method=prf_convolution).


... (28 outputs left out)

[2026-07-25T15:55:42][koa_middleware.database.local_database:245][INFO]: Calibration ID '338744c9-a6c6-41d9-8492-9247a0d0f35c' not found in table 'hispec'.


[2026-07-25T15:55:42][koa_middleware.store:274][WARNING]: Calibration already exists in cache: LFCWavesolModel()! Skipping registration.


[2026-07-25T15:55:42][hispecdrp.simulations.simulate_up_the_ramp:259][INFO]: Registered simulated LFC wavesol in caldb (spectrograph=BSPEC): None


ramp type: RampModel
ramp filename: simulated_ramp.fits
ramp data shape: (10, 4096, 4160)
Stage-0 cals: ['bias', 'dark', 'detflat', 'dq', 'gain', 'rn', 'saturation']
rate_map shape: (4096, 4160)


## 3. Stage 0 → 1 → 2 via pipeline classes

Primitives resolve calibrations from the KOA caldb using the ramp / L0 / L1 metadata.


In [4]:
from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline

# Stage 0 ramp-fit flags non-finite rate/rate_err as DO_NOT_USE; Stage 1 extraction
# also drops non-finite rate/err in the aperture mask. Keep the box on the profile core.
stage1_args = {"spec_extract": {"extraction_aperture": 5}}

l0 = Stage0Pipeline(output_dir=OUT).apply(ramp)

l1 = Stage1Pipeline(output_dir=OUT, prim_args=stage1_args).apply(l0)

# The template is stamped at 0 km/s topocentric (no barycentric shift), so disable the
# CCF barycentric term and center the search at 0. auto_center_scout off avoids spurious
# peak locks (the source of the old fiber-2 artifact).
rv_args = {
    "save_preview": True,
    "preview_write_png": False,
    "ccf_input_velocity": 0.0,
    "ccf_zb_from_barycorr": False,
    "mask_auto_center_scout": False,
}
l2 = Stage2Pipeline(output_dir=OUT, prim_args={"stellar_rv": rv_args}).apply(l1)

print(f"L0 type: {l0.meta.model_type}")
print(f"L0 filename: {l0.meta.filename}")
print(f"L1 type: {l1.meta.model_type}")
print(f"L1 filename: {l1.meta.filename}")
print(f"L2 type: {l2.meta.model_type}")
print(f"L2 filename: {l2.meta.filename}")


[2026-07-25T15:55:42][hispecdrp.pipelines.base_pipeline:130][INFO]: Applying Stage0Pipeline with arguments:
input = RampModel('simulated_ramp.fits')
run = True
save_result = True
output_model = input
output_file = None
output_dir = /Users/ckilpatrick/Downloads/hispec_pipeline_example
suffix = None
save_preview = False
show_preview = False


[2026-07-25T15:55:42][hispecdrp.primitives.base_primitive:282][INFO]: Applying DQInitPrimitive with arguments:
input = RampModel('simulated_ramp.fits')
run = True
save_result = None
output_model = input
output_file = None
output_dir = None
suffix = dqinit
save_preview = False
show_preview = False
max_cores = 1
dq = {'selector': <class 'hispecdrp.calibrations.selectors.DQSelector'>, 'selector_kwargs': {}}


[2026-07-25T15:55:42][koa_middleware.store:186][INFO]: Cache initialized: cache_dir='/Users/ckilpatrick/Downloads/hispec_pipeline_example/cals/KOA_CALIBRATION_CACHE', data_dir='/Users/ckilpatrick/Downloads/hispec_pipeline_example/cals/KOA_CALIBRATION_CACHE/calibrations/hispec/', db='/Users/ckilpatrick/Downloads/hispec_pipeline_example/cals/KOA_CALIBRATION_CACHE/database/hispec_calibrations.db', table='hispec'


[2026-07-25T15:55:42][koa_middleware.database.remote_database:35][INFO]: RemoteCalibrationDB initialized: instrument='hispec', url='https://www3.keck.hawaii.edu/api/calibrations'


[2026-07-25T15:55:42][koa_middleware.store:808][INFO]: Syncing records from remote DB (mode='id')...


[2026-07-25T15:55:42][koa_middleware.database.remote_database:146][INFO]: Querying remote DB at 'https://www3.keck.hawaii.edu/api/calibrations/hispec/query' with params={}


... (377 outputs left out)

[2026-07-25T15:56:31][hispecdrp.primitives.stellar_rv.previews.stellar_rv_preview:3457][INFO]: Stellar RV preview: create_stellar_rv_preview finished


[2026-07-25T15:56:31][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:1480][INFO]: Stellar RV: _create_preview finished


[2026-07-25T15:56:31][hispecdrp.processing_component:154][INFO]: Saved L2Model('HB.20240924.00000.00_spec.fits')


L0 type: L0Model
L0 filename: HB.20240924.00000.00_rate.fits
L1 type: L1Model
L1 filename: HB.20240924.00000.00_specuncal.fits
L2 type: L2Model
L2 filename: HB.20240924.00000.00_spec.fits


## 4. Stellar RV preview


In [5]:
from IPython.display import HTML, display
from pathlib import Path
import html

preview = l2.get_output_path(output_dir=OUT, suffix="stellar_rv").replace(".fits", "_preview.html")
print(f"{preview} ({os.path.getsize(preview) / 1e6:.2f} MB)")
# iframe srcdoc so Plotly JS runs inside the notebook
display(HTML(
    f'<iframe srcdoc="{html.escape(Path(preview).read_text(encoding="utf-8"), quote=True)}" '
    f'width="100%" height="1400" style="border:0" sandbox="allow-scripts allow-same-origin"></iframe>'
))


/Users/ckilpatrick/Downloads/hispec_pipeline_example/HB.20240924.00000.00_spec-stellar_rv_preview.html (0.32 MB)


/Users/ckilpatrick/scripts/anaconda3/envs/hispec/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## 5. High-precision RV (line-by-line CCF)

Line-by-line windowing on a fine velocity grid recovers the injected 0 km/s to ~1 m/s in the noiseless limit. At `J=3` with realistic noise the formal per-fiber precision is a few m/s.

In [6]:
# High-precision RV: line-by-line CCF on a fine grid. Recovers the injected 0 km/s to
# ~1 m/s in the noiseless limit; with noise at J=3 the formal per-fiber precision is a
# few m/s. Kept separate from the preview above (line-by-line makes a much larger HTML).
hp_args = {
    **rv_args,
    "save_preview": False,
    "window_line_by_line": True,
    "v_half_window": 5000.0,  # m/s
    "v_step_coarse": 25.0,  # m/s
    "chunk_rv_prior_half_width_m_s": 3000.0,
    "mask_ccf_pixel_interp_order": 1,
}
l2_hp = Stage2Pipeline(output_dir=OUT, prim_args={"stellar_rv": hp_args}).apply(l1)
for f in (1, 2, 3):
    avg = getattr(l2_hp, f"rvccfavg_fib{f}")
    print(f"fiber {f}: RV = {float(avg['RV'][0]):+7.2f} +/- {float(avg['RV_ERR'][0]):.2f} m/s")

[2026-07-25T15:56:31][hispecdrp.pipelines.base_pipeline:130][INFO]: Applying Stage2Pipeline with arguments:
input = L1Model('HB.20240924.00000.00_specuncal.fits')
run = True
save_result = True
output_model = input
output_file = None
output_dir = /Users/ckilpatrick/Downloads/hispec_pipeline_example
suffix = None
save_preview = False
show_preview = False


[2026-07-25T15:56:31][hispecdrp.primitives.base_primitive:282][INFO]: Applying BlazeCorrectionPrimitive with arguments:
input = L1Model('HB.20240924.00000.00_specuncal.fits')
run = True
save_result = None
output_model = L2Model
output_file = None
output_dir = None
suffix = blaze_correct
save_preview = False
show_preview = False
max_cores = 1
do_fibers = None
do_orders = None
remove_lamp_blackbody = False
normalize_for_stitching = False
pixels_per_spline_knot = 16
fiberflat = {'selector': <class 'hispecdrp.calibrations.selectors.FiberFlatSelector'>, 'selector_kwargs': {}}


[2026-07-25T15:56:31][koa_middleware.store:186][INFO]: Cache initialized: cache_dir='/Users/ckilpatrick/Downloads/hispec_pipeline_example/cals/KOA_CALIBRATION_CACHE', data_dir='/Users/ckilpatrick/Downloads/hispec_pipeline_example/cals/KOA_CALIBRATION_CACHE/calibrations/hispec/', db='/Users/ckilpatrick/Downloads/hispec_pipeline_example/cals/KOA_CALIBRATION_CACHE/database/hispec_calibrations.db', table='hispec'


[2026-07-25T15:56:31][koa_middleware.database.remote_database:35][INFO]: RemoteCalibrationDB initialized: instrument='hispec', url='https://www3.keck.hawaii.edu/api/calibrations'


[2026-07-25T15:56:31][koa_middleware.store:808][INFO]: Syncing records from remote DB (mode='id')...


[2026-07-25T15:56:31][koa_middleware.database.remote_database:146][INFO]: Querying remote DB at 'https://www3.keck.hawaii.edu/api/calibrations/hispec/query' with params={}


... (66 outputs left out)

[2026-07-25T16:00:11][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:1013][INFO]: Stellar RV: RV tables written fiber=3 (RVCCF + RVCCFAVG)


[2026-07-25T16:00:11][hispecdrp.primitives.stellar_rv.stellar_rv_primitive:436][INFO]: Stellar RV: _perform finished algorithm=ccf_line_mask


[2026-07-25T16:00:16][hispecdrp.processing_component:154][INFO]: Saved L2Model('HB.20240924.00000.00_spec.fits')


fiber 1: RV =  -35.25 +/- 3.73 m/s
fiber 2: RV =  -48.42 +/- 3.94 m/s
fiber 3: RV =  -67.25 +/- 3.82 m/s
